# 37 GPU audio separation and embeddings

音声分離 branch と Hugging Face audio transformer embedding branch を Colab GPU で作ります。HF token は登録済み前提です。

Creates: `predictions/{audio_separated_run_id}/`, `predictions/{audio_embedding_run_id}/`, `features/{audio_separated_feature_id}/`, `features/{audio_embedding_feature_id}/`, `reports/audio_impact/...`。

Required inputs: 36 と同じ `bbe_events_v1` / `clips_v1`。Demucs を使う場合は clip files の audio track が必要です。

Next: `38_cpu_audio_fusion_compare.ipynb`.

In [ ]:
from pathlib import Path
import json
import os
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or already mounted:', exc)

os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff_with_audio')
repo_dir = Path(os.environ['BATTING_CODE_ROOT'])
if not (repo_dir / 'src' / 'sport_pipeline').exists():
    repo_dir = Path.cwd()
sys.path.insert(0, str(repo_dir / 'src'))

profile = json.loads((repo_dir / 'configs' / 'runs' / os.environ['BASEBALL_VISION_RUN_PROFILE']).read_text(encoding='utf-8'))
BASE_DIR = Path(profile['paths']['base_dir'])
CACHE_DIR = Path(profile['paths'].get('cache_dir', '/content/cache/baseball_vision'))
RUN_IDS = profile['run_ids']
NS = profile['artifact_namespace']
SEP_EXEC = profile['execution']['audio_separation']
EMB_EXEC = profile['execution']['audio_embedding']
print('BASE_DIR=', BASE_DIR)
print('GPU branch max clips:', SEP_EXEC.get('max_clips'), EMB_EXEC.get('max_clips'))

In [ ]:
INSTALL_DEPS = True
if INSTALL_DEPS:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'soundfile', 'librosa'])
    if SEP_EXEC.get('separation_backend') == 'demucs':
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'demucs'])

In [ ]:
from sport_pipeline.audio.separation import run_audio_separation_experiment
from sport_pipeline.audio.embeddings import run_hf_audio_embedding_baseline

outputs = {}
if SEP_EXEC.get('execute_default', True):
    outputs['separated'] = run_audio_separation_experiment(
        BASE_DIR,
        clip_run_id=RUN_IDS['full_run_id'],
        prediction_run_id=RUN_IDS['audio_separated_run_id'],
        audio_feature_id=NS['audio_separated_feature_id'],
        separation_backend=SEP_EXEC.get('separation_backend', 'transient_enhance'),
        demucs_model=SEP_EXEC.get('demucs_model', 'htdemucs'),
        demucs_stem=SEP_EXEC.get('demucs_stem', 'no_vocals'),
        device=SEP_EXEC.get('device', 'cuda'),
        allow_model_download=SEP_EXEC.get('allow_model_download', True),
        max_clips=SEP_EXEC.get('max_clips'),
        require_non_empty=SEP_EXEC.get('require_non_empty', True),
        cache_dir=CACHE_DIR,
        cache_inputs=False,
    )
if EMB_EXEC.get('execute_default', True):
    outputs['embedding'] = run_hf_audio_embedding_baseline(
        BASE_DIR,
        clip_run_id=RUN_IDS['full_run_id'],
        prediction_run_id=RUN_IDS['audio_embedding_run_id'],
        audio_feature_id=NS['audio_embedding_feature_id'],
        hf_model_id=EMB_EXEC.get('hf_model_id'),
        allow_model_download=EMB_EXEC.get('allow_model_download', True),
        trust_remote_code=EMB_EXEC.get('trust_remote_code', False),
        device=EMB_EXEC.get('device', 'auto'),
        max_clips=EMB_EXEC.get('max_clips'),
        preprocessing_mode=EMB_EXEC.get('preprocessing_mode', 'enhanced'),
        require_non_empty=EMB_EXEC.get('require_non_empty', True),
    )
print(json.dumps({k: {kk: str(vv) for kk, vv in v.items()} for k, v in outputs.items()}, indent=2, ensure_ascii=False))

In [ ]:
for branch, branch_outputs in outputs.items():
    print('\n==', branch, '==')
    for key in ['predictions', 'metrics', 'audio_report_html', 'audio_report_summary']:
        if key in branch_outputs:
            print(key, branch_outputs[key])
print('\nNext notebook: notebooks/38_cpu_audio_fusion_compare.ipynb')